In [ ]:
# ============================================================
# CELL 1 — Install (inference-only)
# ============================================================
!pip -q install -U \
  "transformers>=4.45.0,<5.0.0" \
  "accelerate" \
  "bitsandbytes" \
  "sentencepiece" \
  "tqdm"


In [ ]:
# ============================================================
# CELL 2 — Imports, Drive mount, HF login
# ============================================================
import os, re, json, time, hashlib
from pathlib import Path
import pandas as pd
import torch
from tqdm.auto import tqdm

try:
    from google.colab import drive
    drive.mount("/content/drive")
    IN_COLAB = True
except Exception:
    IN_COLAB = False

# Gemma 3 and RoLlama3 are gated on HuggingFace. Accept the license on the
# model page, then uncomment and paste a token. Qwen3 needs no token.
from huggingface_hub import login
# login("hf_xxx")

print("Torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
# ============================================================
# CELL 3 — Config: model selector + paths + label vocab
# ============================================================
# Pick one: "qwen3-4b" | "rollama3-8b" | "gemma3-4b"
MODEL_CHOICE = "qwen3-4b"

MODEL_REGISTRY = {
    "qwen3-4b":    {"repo": "Qwen/Qwen3-4B-Instruct-2507", "loader": "causal", "load_4bit": False},
    "rollama3-8b": {"repo": "OpenLLM-Ro/RoLlama3-8b-Instruct", "loader": "causal", "load_4bit": True},
    "gemma3-4b":   {"repo": "google/gemma-3-4b-it", "loader": "gemma3", "load_4bit": False},
}
MODEL_REPO = MODEL_REGISTRY[MODEL_CHOICE]["repo"]

BASE_DIR = Path("/content/drive/MyDrive/chiro_note_pipeline") if IN_COLAB else Path("./chiro_note_pipeline")
RAW_CHIRO_CSV_DIR = BASE_DIR / "data" / "raw_chiro_csvs"   # Drive layout used in Colab
LOCAL_RAW_CHIRO_CSV_DIR = Path("data/chiropractor_ro/raw_chiro_csvs")
if not RAW_CHIRO_CSV_DIR.exists() and LOCAL_RAW_CHIRO_CSV_DIR.exists():
    RAW_CHIRO_CSV_DIR = LOCAL_RAW_CHIRO_CSV_DIR
RUN_NAME = time.strftime("%Y%m%d_%H%M%S")
PRED_DIR = BASE_DIR / "predictions" / MODEL_CHOICE / RUN_NAME
PRED_DIR.mkdir(parents=True, exist_ok=True)

MAX_NEW_TOKENS = 700
MAX_SEQ_LENGTH = 2048
LIMIT = None   # set to e.g. 5 for a quick run; None = all conversations

# Closed label sets used in the prompt. TODO: replace with exact locked strings.
LOCALIZARE_REGIUNI = [
    "cervical", "toracal", "lombar", "sacral",
    "umar_dr", "umar_stg", "sold_dr", "sold_stg", "genunchi_dr", "genunchi_stg",
]
ANTECEDENTE_CONDITII = ["hipertensiune", "diabet", "hernia_disc", "scolioza"]

print("Model:", MODEL_REPO)
print("Data dir:", RAW_CHIRO_CSV_DIR)
print("Output:", PRED_DIR)


In [ ]:
# ============================================================
# CELL 4 — Prompt (RO, strict) + per-model chat template
# ============================================================
SYSTEM_PROMPT = f"""Ești un asistent medical care extrage informații dintr-o conversație \
între un chiropractician și un pacient și completează o fișă structurată în format JSON.

REGULĂ ABSOLUTĂ: dacă o informație NU este rostită explicit în conversație, lași câmpul gol. \
NU inventezi, NU deduci, NU completezi din context probabil. Absent în conversație → gol.

Răspunzi DOAR cu un obiect JSON valid, fără text înainte sau după, cu exact aceste câmpuri:

- "motivul_prezentarii": text liber, motivul principal (string, "" dacă nespus)
- "evaluarea_durerii_vas": număr întreg 0-10 doar dacă pacientul rostește un număr; altfel null
- "localizarea_durerii": listă din: {LOCALIZARE_REGIUNI}; [] dacă niciuna rostită
- "antecedente": obiect {{"conditii": listă din {ANTECEDENTE_CONDITII}, "altele": text liber}}
- "medicatie_actuala": listă de {{"nume": string, "doza": string sau null}}; [] dacă niciuna
- "evaluare_functionala_initiala": observații funcționale rostite de terapeut; null dacă nespus
"""

USER_TEMPLATE = "Conversație:\n{dialogue}\n\nFișă JSON:"


def build_messages(dialogue):
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": USER_TEMPLATE.format(dialogue=dialogue)},
    ]


def apply_chat_template_safe(tokenizer, messages, model_choice):
    if model_choice == "gemma3-4b":
        # Gemma 3 has no system role: fold system into first user turn.
        sys_text, merged = "", []
        for m in messages:
            if m["role"] == "system":
                sys_text = m["content"]; continue
            if m["role"] == "user" and sys_text:
                merged.append({"role": "user", "content": sys_text + "\n\n" + m["content"]}); sys_text = ""
            else:
                merged.append(m)
        return tokenizer.apply_chat_template(merged, tokenize=False, add_generation_prompt=True)
    if model_choice == "qwen3-4b":
        # Disable thinking so no <think> blocks corrupt the JSON.
        return tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


In [ ]:
# ============================================================
# CELL 5 — Load model + tokenizer
# ============================================================
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

spec = MODEL_REGISTRY[MODEL_CHOICE]
tokenizer = AutoTokenizer.from_pretrained(MODEL_REPO)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

load_kwargs = {"device_map": "auto"}
if spec["load_4bit"]:
    load_kwargs["quantization_config"] = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True)
else:
    load_kwargs["torch_dtype"] = torch.float16

if spec["loader"] == "gemma3":
    # gemma-3-4b-it is multimodal; use the conditional-generation class for text.
    try:
        from transformers import Gemma3ForConditionalGeneration
        model = Gemma3ForConditionalGeneration.from_pretrained(MODEL_REPO, **load_kwargs)
    except Exception as e:
        print("Gemma3ForConditionalGeneration unavailable, falling back:", repr(e))
        model = AutoModelForCausalLM.from_pretrained(MODEL_REPO, **load_kwargs)
else:
    model = AutoModelForCausalLM.from_pretrained(MODEL_REPO, **load_kwargs)

model.eval()
print("Loaded:", MODEL_REPO, "| 4-bit:", spec["load_4bit"], "| device:", next(model.parameters()).device)


In [ ]:
# ============================================================
# CELL 6 — Load REAL data from chiropractor CSVs
# ============================================================
# CSV columns: dialogue, optionally conversation_id. One conversation may
# span several section rows -> group by conversation_id/dialogue, keep one row per conversation.
def stable_id(text):
    return "conv_" + hashlib.sha1(text.encode("utf-8")).hexdigest()[:12]

def load_real_dialogues(folder):
    folder = Path(folder)
    if not folder.exists():
        raise FileNotFoundError(f"Missing folder: {folder}")
    csvs = sorted(folder.glob("*.csv"))
    print(f"Found {len(csvs)} CSV(s)")
    rows = []
    for p in csvs:
        df = pd.read_csv(p)
        if "dialogue" not in df.columns:
            print("  skip (no dialogue column):", p.name); continue
        df = df.dropna(subset=["dialogue"]).copy()
        if "conversation_id" not in df.columns:
            df["conversation_id"] = df["dialogue"].map(lambda x: stable_id(str(x).strip()))
        for _, r in df.drop_duplicates("conversation_id").iterrows():
            dialogue = str(r["dialogue"]).strip()
            if dialogue:
                rows.append({"conversation_id": str(r["conversation_id"]).strip(),
                             "source_file": p.name, "dialogue": dialogue})
    out = pd.DataFrame(rows).drop_duplicates("conversation_id").reset_index(drop=True)
    print("Conversations:", len(out))
    return out

data_df = load_real_dialogues(RAW_CHIRO_CSV_DIR)
display(data_df.head())


In [ ]:
# ============================================================
# CELL 7 — Run model over real data + export
# ============================================================
def strip_think(t):  return re.sub(r"<think>.*?</think>", "", t, flags=re.DOTALL | re.IGNORECASE).strip()
def strip_fences(t):
    t = re.sub(r"^```(?:json)?\s*", "", t.strip(), flags=re.IGNORECASE)
    return re.sub(r"\s*```$", "", t).strip()

def extract_json(text):
    text = strip_fences(strip_think(text))
    start = text.find("{")
    if start < 0: return None
    depth = 0; in_str = False; esc = False
    for i in range(start, len(text)):
        ch = text[i]
        if in_str:
            if esc: esc = False
            elif ch == "\\": esc = True
            elif ch == '"': in_str = False
            continue
        if ch == '"': in_str = True
        elif ch == "{": depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0:
                return text[start:i+1]
    return None

def generate_raw(dialogue):
    prompt = apply_chat_template_safe(tokenizer, build_messages(dialogue), MODEL_CHOICE)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True,
                       max_length=MAX_SEQ_LENGTH).to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS,
                             do_sample=False, num_beams=1,
                             pad_token_id=tokenizer.pad_token_id,
                             eos_token_id=tokenizer.eos_token_id)
    gen = out[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(gen, skip_special_tokens=True).strip()

work_df = data_df.head(LIMIT) if LIMIT else data_df
out_jsonl = PRED_DIR / "predictions.jsonl"
records = []

for _, row in tqdm(work_df.iterrows(), total=len(work_df), desc="Predict"):
    raw = generate_raw(row["dialogue"])
    block = extract_json(raw)
    note = None
    if block:
        try: note = json.loads(block)
        except json.JSONDecodeError: note = None
    rec = {
        "run_name": RUN_NAME, "model_choice": MODEL_CHOICE, "model_repo": MODEL_REPO,
        "conversation_id": row["conversation_id"], "source_file": row["source_file"],
        "dialogue": row["dialogue"], "prediction_note": note, "raw_output": raw,
    }
    records.append(rec)
    with out_jsonl.open("a", encoding="utf-8") as f:   # incremental in case Colab drops
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

pred_df = pd.DataFrame(records)
csv_df = pred_df.copy()
csv_df["prediction_note"] = csv_df["prediction_note"].apply(lambda x: json.dumps(x, ensure_ascii=False))
csv_df.to_csv(PRED_DIR / "predictions.csv", index=False)

print("Saved:", out_jsonl)
print("Parsed-OK rate:", pred_df["prediction_note"].notna().mean())
display(pred_df[["conversation_id", "prediction_note"]].head())
